# 05 — Final Evaluation on Test Set

**Purpose:** Comprehensive evaluation of the ensemble vs. baseline:
1. Standard COCO mAP benchmarks
2. Optimal F1 threshold search
3. Strict operational metrics (conf > 0.50)
4. Forensic breakdown: Rescues, Regressions, Both Missed, Ghosts

In [1]:
import os, sys, io, json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

DATA_ROOT = r"C:\Users\dadab\Desktop\Clean Version\data"
OUTPUT_DIR = "outputs/"
IMG_ROOT = os.path.join(DATA_ROOT, "test/images")
GT_PATH = os.path.join(DATA_ROOT, "test/test_annotations.json")
BASELINE_JSON = os.path.join(OUTPUT_DIR, "test_results_exp01_baseline_best.json")
ENSEMBLE_JSON = os.path.join(OUTPUT_DIR, "test_results_ensemble_grand.json")

EVAL_IOU = 0.50
EVAL_CONF = 0.50
GHOST_CONF = 0.85

In [2]:
def get_iou(a, b):
    """IoU for [x,y,w,h] format."""
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[0]+a[2],b[0]+b[2]), min(a[1]+a[3],b[1]+b[3])
    inter = max(0,xB-xA)*max(0,yB-yA)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union > 0 else 0

## 1. COCO mAP + Optimal F1

In [3]:
def coco_and_f1(gt_path, res_path):
    old = sys.stdout; sys.stdout = io.StringIO()
    try:
        gt = COCO(gt_path); dt = gt.loadRes(res_path)
        ev = COCOeval(gt, dt, 'bbox'); ev.evaluate(); ev.accumulate(); ev.summarize()
        ap5095, ap50 = ev.stats[0], ev.stats[1]
    except: ap5095, ap50 = 0, 0
    finally: sys.stdout = old

    with open(res_path) as f: preds = json.load(f)
    total_gt = len(gt.getAnnIds())
    best_f1, best = 0, (0,0,0,0)
    for th in np.arange(0.1, 0.96, 0.05):
        curr = [p for p in preds if p['score'] >= th]
        by_img = {}
        for p in curr: by_img.setdefault(p['image_id'], []).append(p)
        tp, fp = 0, 0
        for iid in gt.getImgIds():
            gts = gt.imgToAnns.get(iid, [])
            dets = sorted(by_img.get(iid, []), key=lambda x: x['score'], reverse=True)
            m = set()
            for d in dets:
                hit = False
                for i, g in enumerate(gts):
                    if i not in m and get_iou(d['bbox'], g['bbox']) >= EVAL_IOU:
                        m.add(i); tp += 1; hit = True; break
                if not hit: fp += 1
        p = tp/(tp+fp) if tp+fp > 0 else 0
        r = tp/total_gt if total_gt > 0 else 0
        f1 = 2*p*r/(p+r) if p+r > 0 else 0
        if f1 > best_f1: best_f1 = f1; best = (p, r, f1, th)
    return (ap5095, ap50), best

cb, ob = coco_and_f1(GT_PATH, BASELINE_JSON)
ce, oe = coco_and_f1(GT_PATH, ENSEMBLE_JSON)

print("=" * 70)
print("SCIENTIFIC BENCHMARKS")
print("=" * 70)
print(f"{'Metric':<20} | {'Baseline':<20} | {'Ensemble':<20}")
print("-" * 70)
print(f"{'mAP@50:95':<20} | {cb[0]:.4f}               | {ce[0]:.4f}")
print(f"{'AP@50':<20} | {cb[1]:.4f}               | {ce[1]:.4f}")
print(f"{'Max F1':<20} | {ob[2]:.4f} (Th:{ob[3]:.2f})    | {oe[2]:.4f} (Th:{oe[3]:.2f})")

SCIENTIFIC BENCHMARKS
Metric               | Baseline             | Ensemble            
----------------------------------------------------------------------
mAP@50:95            | 0.5420               | 0.5598
AP@50                | 0.7958               | 0.8033
Max F1               | 0.7877 (Th:0.75)    | 0.7826 (Th:0.90)


## 2. Strict Operational Metrics (conf > 0.50)

In [4]:
def strict_metrics(gt_path, res_path):
    coco = COCO(gt_path)
    with open(res_path) as f: preds = json.load(f)
    hi = [p for p in preds if p['score'] > EVAL_CONF]
    avg_c = np.mean([p['score'] for p in hi]) if hi else 0
    by_img = {}
    for p in hi: by_img.setdefault(p['image_id'], []).append(p)
    tp, fp, fn = 0, 0, 0
    for iid in coco.getImgIds():
        gts = coco.imgToAnns.get(iid, [])
        dets = sorted(by_img.get(iid, []), key=lambda x: x['score'], reverse=True)
        m = set()
        for d in dets:
            best_iou, best_j = 0, -1
            for j, g in enumerate(gts):
                if j not in m:
                    iou = get_iou(d['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= EVAL_IOU: m.add(best_j); tp += 1
            else: fp += 1
        fn += len(gts) - len(m)
    p = tp/(tp+fp) if tp+fp>0 else 0
    r = tp/(tp+fn) if tp+fn>0 else 0
    f = 2*p*r/(p+r) if p+r>0 else 0
    return p, r, f, avg_c

pb, rb, fb, cb = strict_metrics(GT_PATH, BASELINE_JSON)
pe, re, fe, ce_ = strict_metrics(GT_PATH, ENSEMBLE_JSON)

print("\n" + "=" * 70)
print("OPERATIONAL METRICS (Confidence > 0.50)")
print("=" * 70)
print(f"{'Metric':<15} | {'Baseline':<10} | {'Ensemble':<10} | {'Delta':<10}")
print("-" * 55)
print(f"{'Recall':<15} | {rb:.4f}     | {re:.4f}     | {re-rb:+.4f}")
print(f"{'Precision':<15} | {pb:.4f}     | {pe:.4f}     | {pe-pb:+.4f}")
print(f"{'F1-Score':<15} | {fb:.4f}     | {fe:.4f}     | {fe-fb:+.4f}")
print(f"{'Avg Conf':<15} | {cb:.4f}     | {ce_:.4f}     | {ce_-cb:+.4f}")

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!

OPERATIONAL METRICS (Confidence > 0.50)
Metric          | Baseline   | Ensemble   | Delta     
-------------------------------------------------------
Recall          | 0.8317     | 0.9089     | +0.0773
Precision       | 0.7125     | 0.5457     | -0.1668
F1-Score        | 0.7675     | 0.6820     | -0.0855
Avg Conf        | 0.8984     | 0.8690     | -0.0294


## 3. Forensic Breakdown

In [5]:
coco = COCO(GT_PATH)
def load_preds(path, thr):
    with open(path) as f: d = json.load(f)
    m = {}
    for p in d:
        if p['score'] >= thr: m.setdefault(p['image_id'], []).append(p)
    return m

base_p = load_preds(BASELINE_JSON, EVAL_CONF)
ens_p = load_preds(ENSEMBLE_JSON, EVAL_CONF)
ens_ghost = load_preds(ENSEMBLE_JSON, GHOST_CONF)

total_gt, base_hits, ens_hits = 0, 0, 0
rescues, regressions, both_missed, ghosts = 0, 0, 0, 0

for iid in coco.getImgIds():
    gts = coco.imgToAnns.get(iid, [])
    total_gt += len(gts)
    bp, ep, egp = base_p.get(iid,[]), ens_p.get(iid,[]), ens_ghost.get(iid,[])

    for gt in gts:
        bh = any(get_iou(gt['bbox'], p['bbox']) >= EVAL_IOU for p in bp)
        eh = any(get_iou(gt['bbox'], p['bbox']) >= EVAL_IOU for p in ep)
        if bh: base_hits += 1
        if eh: ens_hits += 1
        if not bh and eh: rescues += 1
        elif bh and not eh: regressions += 1
        elif not bh and not eh: both_missed += 1

    for p in egp:
        if not any(get_iou(p['bbox'], g['bbox']) > 0.1 for g in gts):
            ghosts += 1

print("\n" + "=" * 70)
print("FORENSIC BREAKDOWN")
print("=" * 70)
print(f"Total GT: {total_gt}")
print(f"Baseline Hits: {base_hits} ({base_hits/total_gt*100:.1f}%)")
print(f"Ensemble Hits: {ens_hits} ({ens_hits/total_gt*100:.1f}%)")
print(f"{'':->50}")
print(f"RESCUES:     {rescues}")
print(f"REGRESSIONS: {regressions}")
print(f"BOTH MISSED: {both_missed}")
print(f"GHOSTS:      {ghosts} (conf > {GHOST_CONF})")

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!

FORENSIC BREAKDOWN
Total GT: 1812
Baseline Hits: 1537 (84.8%)
Ensemble Hits: 1667 (92.0%)
--------------------------------------------------
RESCUES:     135
REGRESSIONS: 5
BOTH MISSED: 140
GHOSTS:      168 (conf > 0.85)
